[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C20_Frontier_Architectures_Course/04_normalization/04_normalization.ipynb)

# 04 · RMSNorm 与稳定性（从零实现）

目标：手写 LayerNorm/RMSNorm 的前向与反向（数值梯度校验），模拟残差范数增长，演示 QK-Norm 与 z-loss。

路线：LN/RMS 前向 → RMS 反向 → 数值梯度校验 → 残差增长 → QK-Norm → z-loss → ✏️ 练习 → 🧪 胶囊。

## 1 · LayerNorm 与 RMSNorm 前向

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def layernorm(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    var = ((x - mu) ** 2).mean(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps) * g + b

def rmsnorm(x, g, eps=1e-5):
    rms = np.sqrt((x ** 2).mean(-1, keepdims=True) + eps)
    return x / rms * g

d = 8
x = rng.standard_normal(d) * 3 + 1   # 故意带均值和大尺度
g = np.ones(d); b = np.zeros(d)
print('LN 后均值≈0:', round(layernorm(x,g,b).mean(), 6), ' 方差≈1:', round(layernorm(x,g,b).var(), 4))
print('RMSNorm 后 RMS≈1:', round(np.sqrt((rmsnorm(x,g)**2).mean()), 4))
assert abs(layernorm(x,g,b).mean()) < 1e-5
assert abs(np.sqrt((rmsnorm(x,g)**2).mean()) - 1) < 1e-3

## 2 · RMSNorm 反向传播

`dL/dx = (1/r)(g⊙dy − x·(Σ g·dy·x)/(d·r²))`。先实现解析梯度。

In [ ]:
def rmsnorm_backward(x, g, dy, eps=1e-5):
    d = x.shape[-1]
    ms = (x ** 2).mean(-1, keepdims=True) + eps
    r = np.sqrt(ms)
    gdy = g * dy
    dot = (gdy * x).sum(-1, keepdims=True)
    dx = (gdy - x * dot / (d * ms)) / r
    dg = (dy * x / r)
    return dx, dg

x = rng.standard_normal(d)
g = rng.standard_normal(d)
dy = rng.standard_normal(d)
dx, dg = rmsnorm_backward(x, g, dy)
print('dx 形状', dx.shape, 'dg 形状', dg.shape)
assert dx.shape == (d,) and dg.shape == (d,)

## 3 · 数值梯度校验

用有限差分核对解析梯度（标量损失 L = Σ dy·rmsnorm(x)）。

In [ ]:
def num_grad(f, x, eps=1e-6):
    g = np.zeros_like(x)
    for i in range(x.size):
        xp = x.copy(); xp.flat[i] += eps
        xm = x.copy(); xm.flat[i] -= eps
        g.flat[i] = (f(xp) - f(xm)) / (2 * eps)
    return g

L = lambda xx: (dy * rmsnorm(xx, g)).sum()
gx_num = num_grad(L, x)
print('解析 vs 数值 dx 最大误差:', round(np.abs(dx - gx_num).max(), 9))
assert np.allclose(dx, gx_num, atol=1e-5), '梯度校验应通过'
print('✅ RMSNorm 反向梯度正确')

## 4 · Pre-LN vs Post-LN：残差范数随深度

模拟一个 L 层残差堆叠，看激活范数怎么长。

In [ ]:
def simulate_residual(L=40, d=64, mode='pre', seed=0):
    rg = np.random.default_rng(seed)
    x = rg.standard_normal(d)
    g = np.ones(d)
    norms = []
    for _ in range(L):
        W = rg.standard_normal((d, d)) / np.sqrt(d)
        if mode == 'pre':
            x = x + np.tanh(rmsnorm(x, g) @ W)      # 残差是干净高速路
        else:  # post
            x = rmsnorm(x + np.tanh(x @ W), g)       # 每层都归一化整条残差
        norms.append(np.linalg.norm(x))
    return np.array(norms)

pre = simulate_residual(mode='pre')
post = simulate_residual(mode='post')
print('Pre-LN  末层范数:', round(pre[-1], 2), '（随深度累积增长）')
print('Post-LN 末层范数:', round(post[-1], 2), '（被反复拉回，基本平稳）')
assert pre[-1] > pre[0], 'Pre-LN 范数应随深度增长'

> Pre-LN 残差范数随深度增长（解释了为何超深模型常加最终 LN / LayerScale）；Post-LN 平稳但深层梯度更难。

## 5 · QK-Norm：把 attention logit 钉住

对 Q、K 归一化后，点积分数的方差不再随特征尺度爆炸。

In [ ]:
def logit_stats(scale):
    T, dh = 64, 32
    Q = rng.standard_normal((T, dh)) * scale
    K = rng.standard_normal((T, dh)) * scale
    raw = (Q @ K.T) / np.sqrt(dh)
    Qn = Q / np.linalg.norm(Q, axis=-1, keepdims=True)
    Kn = K / np.linalg.norm(K, axis=-1, keepdims=True)
    qk = (Qn @ Kn.T)   # 余弦相似度，范围[-1,1]
    return raw.std(), qk.std()

prev_raw = None
for sc in [1, 3, 10]:
    r, q = logit_stats(sc)
    print(f'激活尺度 x{sc:2d}: 原始 logit 标准差 {r:8.2f} | QK-Norm 后 {q:.3f}（与尺度无关）')
    prev_raw = r
r1, q1 = logit_stats(1); r10, q10 = logit_stats(10)
assert r10 > r1 * 3, '原始 logit 随尺度暴涨'
assert abs(q10 - q1) < 0.1, 'QK-Norm 后几乎不变'
print('✅ QK-Norm 把分数尺度钉住')

## 6 · z-loss：约束 softmax 归一化项

In [ ]:
def z_loss(logits, alpha=1e-4):
    Z = np.exp(logits - logits.max(-1, keepdims=True)).sum(-1)  # 数值稳定
    logZ = np.log(Z) + logits.max(-1)
    return alpha * (logZ ** 2).mean()

small = rng.standard_normal((4, 100))
big = small + 10                 # logits 整体漂移
print('正常 logits 的 z-loss:', round(z_loss(small), 5))
print('漂移 logits 的 z-loss:', round(z_loss(big), 5), '（更大 -> 惩罚漂移）')
assert z_loss(big) > z_loss(small)

---
## ✏️ 练习 1：实现 RMSNorm 前向

实现 `my_rmsnorm(x, g, eps)`，验证输出 RMS≈1、且对输入缩放不变。

In [ ]:
def my_rmsnorm(x, g, eps=1e-5):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
xx = rng.standard_normal(16) * 5
gg = np.ones(16)
y = my_rmsnorm(xx, gg)
assert abs(np.sqrt((y**2).mean()) - 1) < 1e-3, 'RMS 应≈1'
assert np.allclose(my_rmsnorm(xx, gg), my_rmsnorm(2*xx, gg), atol=1e-4), '应对输入缩放不变'
print('✅ 练习 1 通过')

## ✏️ 练习 2：RMSNorm 反向 + 数值校验

实现 `my_rmsnorm_backward(x, g, dy)` 返回 `dx`，用有限差分校验。

In [ ]:
def my_rmsnorm_backward(x, g, dy, eps=1e-5):
    # TODO: 返回 dx（见讲解公式）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
x2 = rng.standard_normal(10); g2 = rng.standard_normal(10); dy2 = rng.standard_normal(10)
dx2 = my_rmsnorm_backward(x2, g2, dy2)
Lf = lambda xx: (dy2 * rmsnorm(xx, g2)).sum()
assert np.allclose(dx2, num_grad(Lf, x2), atol=1e-5), '应通过数值梯度校验'
print('✅ 练习 2 通过')

## ✏️ 练习 3：FLOPs 直觉——RMSNorm 省在哪

实现 `norm_reduce_ops(d)` 返回 (LayerNorm, RMSNorm) 沿特征维的“归约次数”（均值+方差算 2 次，RMS 只算 1 次）。

In [ ]:
def norm_reduce_ops(d):
    # TODO: 返回 (ln_reductions, rms_reductions)，LN 需均值与方差两次归约，RMS 一次
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert norm_reduce_ops(4096) == (2, 1)
print('✅ 练习 3 通过：RMSNorm 少一次沿特征维的归约（且无 bias 参数）')

---
### 📖 参考答案

In [ ]:
# 练习 1
def my_rmsnorm(x, g, eps=1e-5):
    return x / np.sqrt((x**2).mean(-1, keepdims=True) + eps) * g

# 练习 2
def my_rmsnorm_backward(x, g, dy, eps=1e-5):
    d = x.shape[-1]
    ms = (x**2).mean(-1, keepdims=True) + eps
    r = np.sqrt(ms)
    gdy = g * dy
    dot = (gdy * x).sum(-1, keepdims=True)
    return (gdy - x * dot / (d * ms)) / r

# 练习 3
def norm_reduce_ops(d):
    return (2, 1)

---
## 🧪 真实数据胶囊：谁用 RMSNorm？

真实模型的归一化选择。

In [ ]:
NORM = {
    'GPT-2':   'LayerNorm (Post->Pre)',
    'Llama-3': 'RMSNorm (Pre)',
    'T5':      'RMSNorm',
    'Gemma':   'RMSNorm (Pre, +最终归一化, +logit soft-cap)',
    'ViT-22B': 'LayerNorm + QK-Norm',
}
for kk, vv in NORM.items():
    print(f'{kk:10s}: {vv}')
print('\n趋势：现代 decoder-only LLM 几乎全用 Pre-RMSNorm；超大模型再叠 QK-Norm/z-loss 稳训练。')

**🧪 胶囊练习**：实现 `rms_param_savings(d, n_layers)` 返回 RMSNorm 相对 LayerNorm 省下的 bias 参数总数（每层 2 个 norm，每个 norm 省 d 个 β）。

In [ ]:
def rms_param_savings(d, n_layers):
    # TODO: 每层 2 个归一化(attn前/ffn前)，每个省 d 个 bias
    raise NotImplementedError

In [ ]:
assert rms_param_savings(4096, 32) == 2*32*4096
print('✅ 胶囊练习通过：Llama-2-7B 量级省下', 2*32*4096, '个 bias 参数（小，但 RMSNorm 主要省的是计算）')

In [ ]:
# 📖 胶囊参考答案
def rms_param_savings(d, n_layers):
    return 2 * n_layers * d

---
### 小结
- RMSNorm = 去均值去 bias 的 LayerNorm，更省、效果相当，现代 LLM 标配。
- Pre-LN 稳、能堆深，但残差范数随深度增长。
- QK-Norm 钉住注意力分数、z-loss 约束输出归一化项——都是“给尺度上软约束”。

下一站：**模块 05 · SSM 与 Mamba**。